# RiskOptima Dislocation Model

This notebook demonstrates `riskoptima.dislocation`. Equities use quality, valuation and a simplified FCF-proxy scenario; funds use historical price/regime dislocation without corporate DCF or a claimed NAV discount. Scores are research heuristics, not calibrated investment forecasts.


### Execution environment

Run from the repository root. Install the released package with `python -m pip install "riskoptima==2.8.4"`. By default this notebook fetches Yahoo Finance data. Set `RISKOPTIMA_DEMO_LIVE=0` before execution for explicitly synthetic, deterministic inputs. This is never a silent fallback for failed live data. Individual ticker failures appear as ERROR rows; an entirely unsuccessful scan fails the notebook.


In [1]:
from IPython.display import Markdown, display
import pandas as pd
import os

from riskoptima.dislocation import (
    DislocationConfig,
    analyze_symbol,
    generate_markdown_report,
    scan_universe,
)

pd.set_option("display.max_columns", 30)
USE_LIVE_DATA = os.environ.get("RISKOPTIMA_DEMO_LIVE", "1") != "0"
if USE_LIVE_DATA:
    provider = None
else:
    from examples.example_dislocation_model import SyntheticProvider

    provider = SyntheticProvider()
print("Data source:", "Yahoo Finance (live)" if USE_LIVE_DATA else "Synthetic, not market data")


Data source: Yahoo Finance (live)


## Input universe

The symbols below are Yahoo identifiers, not ISINs. Yahoo's ISIN search resolves IE00B50W2R13 to `0P0000KM24.L` (Global Bond Index GBP-hedged accumulation share class). Verify share class and currency before replacing an identifier.


In [2]:
INSTRUMENTS = [
    ("0P0000KM24.L", "Global Bond Index Fund GBP Hedged Acc"),
    ("0P0000TKZI.L", "LifeStrategy 40% Equity Fund"),
    ("0P0000TKZO.L", "LifeStrategy 100% Equity Fund"),
    ("0P0000TKZM.L", "LifeStrategy 80% Equity Fund"),
    ("0P00017UWT.L", "Target Retirement 2045 Fund"),
    ("0P0000UGLG.L", "ESG Developed World All Cap Equity Index Fund"),
    ("0P0000UGLE.L", "ESG Developed Europe Index Fund"),
    ("0P0000KM22.L", "Emerging Markets Stock Index Fund"),
    ("0P0000KM20.L", "Pacific ex-Japan Stock Index Fund"),
    ("0P00018XAP.L", "FTSE 100 Index Unit Trust"),
    ("0P0000KSPA.L", "U.S. Equity Index Fund"),
]

pd.DataFrame(INSTRUMENTS, columns=["symbol", "name"])


,symbol,name
0,0P0000KM24.L,Global Bond Index Fund GBP Hedged Acc
1,0P0000TKZI.L,LifeStrategy 40% Equity Fund
2,0P0000TKZO.L,LifeStrategy 100% Equity Fund
3,0P0000TKZM.L,LifeStrategy 80% Equity Fund
4,0P00017UWT.L,Target Retirement 2045 Fund
5,0P0000UGLG.L,ESG Developed World All Cap Equity Index Fund
6,0P0000UGLE.L,ESG Developed Europe Index Fund
7,0P0000KM22.L,Emerging Markets Stock Index Fund
8,0P0000KM20.L,Pacific ex-Japan Stock Index Fund
9,0P00018XAP.L,FTSE 100 Index Unit Trust


## Run and rank all instruments

The engine combines price drawdown, trend deviation, volatility and regime diagnostics. Data coverage measures input completeness, not model accuracy. Markov fitting uses a fixed seed; a fit failure is recorded in diagnostics before the momentum/volatility fallback is used.


In [3]:
config = DislocationConfig(history_period="5y", min_observations=126, use_markov_regime=True)
summary, reports = scan_universe(INSTRUMENTS, config=config, provider=provider, continue_on_error=True)
assert reports, "No successful analyses; inspect the ticker error rows"
print(f"Successful analyses: {len(reports)}/{len(INSTRUMENTS)}")
display(summary.loc[summary["classification"] == "ERROR", ["symbol", "error"]])
summary


Successful analyses: 11/11


,symbol,error


,symbol,name,instrument_type,price,currency,opportunity_score,dislocation_score,quality_score,valuation_score,confidence_score,classification,valuation_mode,error
0,0P0000KM22.L,Emerging Markets Stock Index Fund,MUTUAL_FUND,357.459808,GBP,71.638331,65.895929,None,None,100.0,WATCH,market_dislocation,None
1,0P0000UGLG.L,ESG Developed World All Cap Equity Index Fund,MUTUAL_FUND,563.091797,GBP,49.815418,48.185847,None,None,100.0,NEUTRAL,market_dislocation,None
2,0P0000TKZM.L,LifeStrategy 80% Equity Fund,MUTUAL_FUND,376.325195,GBP,49.601148,44.395747,None,None,100.0,NEUTRAL,market_dislocation,None
3,0P0000TKZO.L,LifeStrategy 100% Equity Fund,MUTUAL_FUND,474.590607,GBP,48.760850,46.436118,None,None,100.0,NEUTRAL,market_dislocation,None
4,0P00017UWT.L,Target Retirement 2045 Fund,MUTUAL_FUND,262.941101,GBP,48.518666,43.092110,None,None,100.0,NEUTRAL,market_dislocation,None
5,0P0000KSPA.L,U.S. Equity Index Fund,MUTUAL_FUND,1234.800537,GBP,46.081850,43.999313,None,None,100.0,NEUTRAL,market_dislocation,None
6,0P0000TKZI.L,LifeStrategy 40% Equity Fund,MUTUAL_FUND,229.063995,GBP,43.459233,38.338842,None,None,100.0,NEUTRAL,market_dislocation,None
7,0P0000KM20.L,Pacific ex-Japan Stock Index Fund,MUTUAL_FUND,450.706390,GBP,39.762385,35.609776,None,None,100.0,EXTENDED,market_dislocation,None
8,0P0000UGLE.L,ESG Developed Europe Index Fund,MUTUAL_FUND,367.375885,GBP,38.891315,28.756644,None,None,100.0,EXTENDED,market_dislocation,None
9,0P00018XAP.L,FTSE 100 Index Unit Trust,MUTUAL_FUND,227.515594,GBP,27.297014,14.085571,None,None,100.0,EXTENDED,market_dislocation,None


## Generate the full report for every successful ticker

The default prose is generated from the measured state and is reproducible. Fund reports disclose optional provider assets, yield, expense ratio, P/E and NAV reference fields, with explicit gaps and limitations. An equity fund has equity exposure, but its units are not corporate shares; these descriptors do not trigger corporate DCF or alter the score. Data coverage is not complete fund valuation coverage. Every report includes a research-only, not-investment-advice notice.

`generate_markdown_report` also accepts a `narrative_generator` callback, so an LLM can be injected by an application without adding an LLM SDK dependency to RiskOptima. The research notice remains present even with custom narratives.


In [4]:
generated_reports = {report.symbol: generate_markdown_report(report) for report in reports}

for symbol, report_markdown in generated_reports.items():
    display(Markdown(report_markdown))
    display(Markdown("---"))


# RiskOptima Dislocation Report — Global Bond Index Fund GBP Hedged Acc

**Ticker:** `0P0000KM24.L`  
**As of:** 2026-11-26  
**Instrument type:** MUTUAL_FUND  
**Valuation mode:** market_dislocation  
**Price:** 159.73 GBP  

## Executive signal

- **Opportunity score:** 21.5/100
- **Classification:** EXTENDED
- **Data coverage:** 100.0/100 (price history/basic identifiers, not complete fund valuation or a probability of accuracy)
- **Dislocation score:** 13.4/100
- **Regime opportunity:** 31.0/100

## Generated interpretation

Global Bond Index Fund GBP Hedged Acc (0P0000KM24.L) scores 21.5/100 and is classified as extended. This is a pooled fund, which may hold equities; its units are not shares in an operating company. RiskOptima does not fabricate a company-style intrinsic value for the fund. Available fund metadata is disclosed separately; the score uses price, drawdown, volatility and regimes. The market-dislocation component is 13.4/100 and the regime-opportunity component is 31.0/100. Data coverage is 100.0/100, not a calibrated probability of model accuracy.

## Market dislocation

| Metric | Value |
|---|---:|
| 1Y drawdown | -1.5% |
| Period drawdown | -2.8% |
| 200D trend deviation | 0.3% |
| Price z-score | 0.64 |
| 3M return | 0.3% |
| 12M return | -0.1% |
| Annualized volatility | 3.8% |
| Volatility percentile | 39.1% |
| Regime | strong |

## Valuation treatment

This instrument is not an operating-company equity. A fund may hold equities, but its units are not corporate shares. RiskOptima deliberately does **not** apply a corporate DCF to the fund itself. The opportunity score measures historical price and regime dislocation, not company intrinsic value. Fund metadata alone does not establish holdings-level fair value; no NAV premium or discount is measured.

## Fund metadata (provider-reported)

| Metric | Value |
|---|---:|
| Total assets (provider units; currency not independently verified) | n/a |
| Distribution yield | 1.7% |
| Annual-report expense ratio | n/a |
| Reported fund P/E (methodology unverified) | n/a |
| NAV reference (date/share class unverified) | n/a |
| Category | n/a |
| Fund family | n/a |

These optional fields are descriptive and do not change the dislocation score. Missing values are shown as n/a. Metadata may differ in date or units from the price history.

## Diagnostics

- Fund metadata coverage: 1/7 optional fields available; missing fields are not estimated.
- Data coverage score reflects price history and basic identifiers, not complete fund valuation data.
- No separate NAV supplied; no NAV premium or discount is measured.
- Fund units may provide equity exposure, but are not corporate shares. Corporate DCF is not applicable to the fund itself; holdings-level fair values and weights would be required for look-through valuation.

## Model note

The score is designed to detect potentially attractive dislocations, not to forecast an exact market bottom. Interpret scores alongside input coverage and, for corporate equities, fundamental quality; neither establishes predictive accuracy.

## Research notice

RiskOptima is for research and educational use only. Reports and scores are not investment advice or recommendations to buy, sell or hold any instrument. Models and third-party data may be incomplete or inaccurate. Past performance does not guarantee future results. Independently verify inputs and results before making investment decisions.

Generated by RiskOptima v2.8.4


---

# RiskOptima Dislocation Report — LifeStrategy 40% Equity Fund

**Ticker:** `0P0000TKZI.L`  
**As of:** 2026-11-26  
**Instrument type:** MUTUAL_FUND  
**Valuation mode:** market_dislocation  
**Price:** 229.06 GBP  

## Executive signal

- **Opportunity score:** 43.5/100
- **Classification:** NEUTRAL
- **Data coverage:** 100.0/100 (price history/basic identifiers, not complete fund valuation or a probability of accuracy)
- **Dislocation score:** 38.3/100
- **Regime opportunity:** 50.1/100

## Generated interpretation

LifeStrategy 40% Equity Fund (0P0000TKZI.L) scores 43.5/100 and is classified as neutral. This is a pooled fund, which may hold equities; its units are not shares in an operating company. RiskOptima does not fabricate a company-style intrinsic value for the fund. Available fund metadata is disclosed separately; the score uses price, drawdown, volatility and regimes. The market-dislocation component is 38.3/100 and the regime-opportunity component is 50.1/100. Data coverage is 100.0/100, not a calibrated probability of model accuracy.

## Market dislocation

| Metric | Value |
|---|---:|
| 1Y drawdown | -2.3% |
| Period drawdown | -2.3% |
| 200D trend deviation | -0.3% |
| Price z-score | -1.84 |
| 3M return | -1.3% |
| 12M return | 3.0% |
| Annualized volatility | 6.2% |
| Volatility percentile | 65.4% |
| Regime | neutral |

## Valuation treatment

This instrument is not an operating-company equity. A fund may hold equities, but its units are not corporate shares. RiskOptima deliberately does **not** apply a corporate DCF to the fund itself. The opportunity score measures historical price and regime dislocation, not company intrinsic value. Fund metadata alone does not establish holdings-level fair value; no NAV premium or discount is measured.

## Fund metadata (provider-reported)

| Metric | Value |
|---|---:|
| Total assets (provider units; currency not independently verified) | 5510755840.00 |
| Distribution yield | 0.0% |
| Annual-report expense ratio | n/a |
| Reported fund P/E (methodology unverified) | 15.22 |
| NAV reference (date/share class unverified) | n/a |
| Category | n/a |
| Fund family | n/a |

These optional fields are descriptive and do not change the dislocation score. Missing values are shown as n/a. Metadata may differ in date or units from the price history.

## Diagnostics

- Fund metadata coverage: 3/7 optional fields available; missing fields are not estimated.
- Data coverage score reflects price history and basic identifiers, not complete fund valuation data.
- Provider-reported fund P/E is descriptive; holdings methodology and valuation date are not verified.
- No separate NAV supplied; no NAV premium or discount is measured.
- Fund units may provide equity exposure, but are not corporate shares. Corporate DCF is not applicable to the fund itself; holdings-level fair values and weights would be required for look-through valuation.

## Model note

The score is designed to detect potentially attractive dislocations, not to forecast an exact market bottom. Interpret scores alongside input coverage and, for corporate equities, fundamental quality; neither establishes predictive accuracy.

## Research notice

RiskOptima is for research and educational use only. Reports and scores are not investment advice or recommendations to buy, sell or hold any instrument. Models and third-party data may be incomplete or inaccurate. Past performance does not guarantee future results. Independently verify inputs and results before making investment decisions.

Generated by RiskOptima v2.8.4


---

# RiskOptima Dislocation Report — LifeStrategy 100% Equity Fund

**Ticker:** `0P0000TKZO.L`  
**As of:** 2026-11-26  
**Instrument type:** MUTUAL_FUND  
**Valuation mode:** market_dislocation  
**Price:** 474.59 GBP  

## Executive signal

- **Opportunity score:** 48.8/100
- **Classification:** NEUTRAL
- **Data coverage:** 100.0/100 (price history/basic identifiers, not complete fund valuation or a probability of accuracy)
- **Dislocation score:** 46.4/100
- **Regime opportunity:** 42.5/100

## Generated interpretation

LifeStrategy 100% Equity Fund (0P0000TKZO.L) scores 48.8/100 and is classified as neutral. This is a pooled fund, which may hold equities; its units are not shares in an operating company. RiskOptima does not fabricate a company-style intrinsic value for the fund. Available fund metadata is disclosed separately; the score uses price, drawdown, volatility and regimes. The market-dislocation component is 46.4/100 and the regime-opportunity component is 42.5/100. Data coverage is 100.0/100, not a calibrated probability of model accuracy.

## Market dislocation

| Metric | Value |
|---|---:|
| 1Y drawdown | -6.1% |
| Period drawdown | -6.1% |
| 200D trend deviation | -0.9% |
| Price z-score | -2.07 |
| 3M return | -3.9% |
| 12M return | 8.8% |
| Annualized volatility | 14.6% |
| Volatility percentile | 79.9% |
| Regime | neutral |

## Valuation treatment

This instrument is not an operating-company equity. A fund may hold equities, but its units are not corporate shares. RiskOptima deliberately does **not** apply a corporate DCF to the fund itself. The opportunity score measures historical price and regime dislocation, not company intrinsic value. Fund metadata alone does not establish holdings-level fair value; no NAV premium or discount is measured.

## Fund metadata (provider-reported)

| Metric | Value |
|---|---:|
| Total assets (provider units; currency not independently verified) | 12857689088.00 |
| Distribution yield | 0.0% |
| Annual-report expense ratio | n/a |
| Reported fund P/E (methodology unverified) | 12.96 |
| NAV reference (date/share class unverified) | n/a |
| Category | n/a |
| Fund family | n/a |

These optional fields are descriptive and do not change the dislocation score. Missing values are shown as n/a. Metadata may differ in date or units from the price history.

## Diagnostics

- Fund metadata coverage: 3/7 optional fields available; missing fields are not estimated.
- Data coverage score reflects price history and basic identifiers, not complete fund valuation data.
- Provider-reported fund P/E is descriptive; holdings methodology and valuation date are not verified.
- No separate NAV supplied; no NAV premium or discount is measured.
- Fund units may provide equity exposure, but are not corporate shares. Corporate DCF is not applicable to the fund itself; holdings-level fair values and weights would be required for look-through valuation.

## Model note

The score is designed to detect potentially attractive dislocations, not to forecast an exact market bottom. Interpret scores alongside input coverage and, for corporate equities, fundamental quality; neither establishes predictive accuracy.

## Research notice

RiskOptima is for research and educational use only. Reports and scores are not investment advice or recommendations to buy, sell or hold any instrument. Models and third-party data may be incomplete or inaccurate. Past performance does not guarantee future results. Independently verify inputs and results before making investment decisions.

Generated by RiskOptima v2.8.4


---

... (12 outputs left out)

# RiskOptima Dislocation Report — FTSE 100 Index Unit Trust

**Ticker:** `0P00018XAP.L`  
**As of:** 2026-11-27  
**Instrument type:** MUTUAL_FUND  
**Valuation mode:** market_dislocation  
**Price:** 227.52 GBP  

## Executive signal

- **Opportunity score:** 27.3/100
- **Classification:** EXTENDED
- **Data coverage:** 100.0/100 (price history/basic identifiers, not complete fund valuation or a probability of accuracy)
- **Dislocation score:** 14.1/100
- **Regime opportunity:** 43.6/100

## Generated interpretation

FTSE 100 Index Unit Trust (0P00018XAP.L) scores 27.3/100 and is classified as extended. This is a pooled fund, which may hold equities; its units are not shares in an operating company. RiskOptima does not fabricate a company-style intrinsic value for the fund. Available fund metadata is disclosed separately; the score uses price, drawdown, volatility and regimes. The market-dislocation component is 14.1/100 and the regime-opportunity component is 43.6/100. Data coverage is 100.0/100, not a calibrated probability of model accuracy.

## Market dislocation

| Metric | Value |
|---|---:|
| 1Y drawdown | -1.8% |
| Period drawdown | -1.8% |
| 200D trend deviation | 2.6% |
| Price z-score | 0.71 |
| 3M return | -0.0% |
| 12M return | 13.6% |
| Annualized volatility | 12.8% |
| Volatility percentile | 59.7% |
| Regime | neutral |

## Valuation treatment

This instrument is not an operating-company equity. A fund may hold equities, but its units are not corporate shares. RiskOptima deliberately does **not** apply a corporate DCF to the fund itself. The opportunity score measures historical price and regime dislocation, not company intrinsic value. Fund metadata alone does not establish holdings-level fair value; no NAV premium or discount is measured.

## Fund metadata (provider-reported)

| Metric | Value |
|---|---:|
| Total assets (provider units; currency not independently verified) | 2211429888.00 |
| Distribution yield | 2.8% |
| Annual-report expense ratio | n/a |
| Reported fund P/E (methodology unverified) | 15.32 |
| NAV reference (date/share class unverified) | n/a |
| Category | n/a |
| Fund family | n/a |

These optional fields are descriptive and do not change the dislocation score. Missing values are shown as n/a. Metadata may differ in date or units from the price history.

## Diagnostics

- Fund metadata coverage: 3/7 optional fields available; missing fields are not estimated.
- Data coverage score reflects price history and basic identifiers, not complete fund valuation data.
- Provider-reported fund P/E is descriptive; holdings methodology and valuation date are not verified.
- No separate NAV supplied; no NAV premium or discount is measured.
- Fund units may provide equity exposure, but are not corporate shares. Corporate DCF is not applicable to the fund itself; holdings-level fair values and weights would be required for look-through valuation.

## Model note

The score is designed to detect potentially attractive dislocations, not to forecast an exact market bottom. Interpret scores alongside input coverage and, for corporate equities, fundamental quality; neither establishes predictive accuracy.

## Research notice

RiskOptima is for research and educational use only. Reports and scores are not investment advice or recommendations to buy, sell or hold any instrument. Models and third-party data may be incomplete or inaccurate. Past performance does not guarantee future results. Independently verify inputs and results before making investment decisions.

Generated by RiskOptima v2.8.4


---

# RiskOptima Dislocation Report — U.S. Equity Index Fund

**Ticker:** `0P0000KSPA.L`  
**As of:** 2026-11-26  
**Instrument type:** MUTUAL_FUND  
**Valuation mode:** market_dislocation  
**Price:** 1234.80 GBP  

## Executive signal

- **Opportunity score:** 46.1/100
- **Classification:** NEUTRAL
- **Data coverage:** 100.0/100 (price history/basic identifiers, not complete fund valuation or a probability of accuracy)
- **Dislocation score:** 44.0/100
- **Regime opportunity:** 39.4/100

## Generated interpretation

U.S. Equity Index Fund (0P0000KSPA.L) scores 46.1/100 and is classified as neutral. This is a pooled fund, which may hold equities; its units are not shares in an operating company. RiskOptima does not fabricate a company-style intrinsic value for the fund. Available fund metadata is disclosed separately; the score uses price, drawdown, volatility and regimes. The market-dislocation component is 44.0/100 and the regime-opportunity component is 39.4/100. Data coverage is 100.0/100, not a calibrated probability of model accuracy.

## Market dislocation

| Metric | Value |
|---|---:|
| 1Y drawdown | -6.9% |
| Period drawdown | -6.9% |
| 200D trend deviation | -0.1% |
| Price z-score | -1.70 |
| 3M return | -2.7% |
| 12M return | 7.0% |
| Annualized volatility | 20.8% |
| Volatility percentile | 76.2% |
| Regime | neutral |

## Valuation treatment

This instrument is not an operating-company equity. A fund may hold equities, but its units are not corporate shares. RiskOptima deliberately does **not** apply a corporate DCF to the fund itself. The opportunity score measures historical price and regime dislocation, not company intrinsic value. Fund metadata alone does not establish holdings-level fair value; no NAV premium or discount is measured.

## Fund metadata (provider-reported)

| Metric | Value |
|---|---:|
| Total assets (provider units; currency not independently verified) | 20786120704.00 |
| Distribution yield | 0.8% |
| Annual-report expense ratio | n/a |
| Reported fund P/E (methodology unverified) | 22.17 |
| NAV reference (date/share class unverified) | n/a |
| Category | n/a |
| Fund family | n/a |

These optional fields are descriptive and do not change the dislocation score. Missing values are shown as n/a. Metadata may differ in date or units from the price history.

## Diagnostics

- Fund metadata coverage: 3/7 optional fields available; missing fields are not estimated.
- Data coverage score reflects price history and basic identifiers, not complete fund valuation data.
- Provider-reported fund P/E is descriptive; holdings methodology and valuation date are not verified.
- No separate NAV supplied; no NAV premium or discount is measured.
- Fund units may provide equity exposure, but are not corporate shares. Corporate DCF is not applicable to the fund itself; holdings-level fair values and weights would be required for look-through valuation.

## Model note

The score is designed to detect potentially attractive dislocations, not to forecast an exact market bottom. Interpret scores alongside input coverage and, for corporate equities, fundamental quality; neither establishes predictive accuracy.

## Research notice

RiskOptima is for research and educational use only. Reports and scores are not investment advice or recommendations to buy, sell or hold any instrument. Models and third-party data may be incomplete or inaccurate. Past performance does not guarantee future results. Independently verify inputs and results before making investment decisions.

Generated by RiskOptima v2.8.4


---

## Single-ticker equity example

For MSFT, equity metadata activates fundamental scoring. The simplified DCF discounts the supplied FCF proxy, without normalized FCFE or net borrowing. Margin of safety is `(model_value - price) / model_value`; upside divides by price instead. Annualized convergence assumes the current model value is reached over the selected horizon, excludes distributions, and is not a return forecast. Missing cash flow or shares results in unavailable DCF fields, not a fabricated value.


In [5]:
equity_report = analyze_symbol("MSFT", config=config, provider=provider)
assert equity_report.instrument_type == "EQUITY"
if equity_report.intrinsic_value is not None:
    assert abs(equity_report.margin_of_safety - (1 - equity_report.price / equity_report.intrinsic_value)) < 1e-10
display(Markdown(generate_markdown_report(equity_report)))
payload = equity_report.as_dict()
assert "riskoptima_version" in payload


# RiskOptima Dislocation Report — Microsoft Corporation

**Ticker:** `MSFT`  
**As of:** 2026-11-27  
**Instrument type:** EQUITY  
**Valuation mode:** fundamental  
**Price:** 492.85 USD  

## Executive signal

- **Opportunity score:** 46.3/100
- **Classification:** NEUTRAL
- **Data coverage:** 100.0/100 (not a probability of accuracy)
- **Dislocation score:** 22.8/100
- **Regime opportunity:** 85.9/100

## Generated interpretation

Microsoft Corporation (MSFT) scores 46.3/100 and is classified as neutral. Fundamental quality is 97.0/100 while the valuation score is 25.9/100. The simplified FCF-proxy scenario value is 9.47, 98.1% below the observed price. This is not a reconciled FCFE valuation. The market-dislocation component is 22.8/100 and the regime-opportunity component is 85.9/100. Data coverage is 100.0/100, not a calibrated probability of model accuracy.

## Market dislocation

| Metric | Value |
|---|---:|
| 1Y drawdown | -11.2% |
| Period drawdown | -11.2% |
| 200D trend deviation | 11.1% |
| Price z-score | 0.40 |
| 3M return | -2.7% |
| 12M return | 2.4% |
| Annualized volatility | 28.1% |
| Volatility percentile | 53.8% |
| Regime | weak |

## Fundamental valuation

| Metric | Value |
|---|---:|
| Quality score | 97.0/100 |
| Valuation score | 25.9/100 |
| FCF yield | 0.1% |
| Earnings yield | 3.9% |
| Forward P/E | 20.34 |
| EV / EBITDA | 18.13 |
| Intrinsic value | 9.47 |
| Margin of safety | -5103.8% |
| Upside to model value | -98.1% |
| Implied 5Y annualized return to model value | -54.6% |
| DCF growth assumption | 12.0% |
| DCF discount rate | 9.0% |
| Terminal growth | 2.5% |

> The intrinsic-value estimate is a scenario model, not a precise fair-price claim. It discounts an unadjusted FCF proxy, not reconciled FCFE; net borrowing is not modeled. Margin of safety is (model value - price) / model value. Upside uses price as the denominator. The annualized figure assumes convergence to today's model value over the stated horizon, excludes distributions, and is not an expected-return forecast. Sensitivity to cash flow, growth and discount rates is material.

## Diagnostics

- DCF uses an unadjusted free-cash-flow proxy; net borrowing and normalized FCFE are not modeled.

## Model note

The score is designed to detect potentially attractive dislocations, not to forecast an exact market bottom. Interpret scores alongside input coverage and, for corporate equities, fundamental quality; neither establishes predictive accuracy.

## Research notice

RiskOptima is for research and educational use only. Reports and scores are not investment advice or recommendations to buy, sell or hold any instrument. Models and third-party data may be incomplete or inaccurate. Past performance does not guarantee future results. Independently verify inputs and results before making investment decisions.

Generated by RiskOptima v2.8.4


## Optional LLM narrative adapter

Applications can inject any callable accepting the report context and returning text. This keeps the quantitative model deterministic while allowing the final narrative to be genuinely generative.

```python
def my_generator(context: dict) -> str:
    # Call your preferred model/service here.
    return "..."

report = analyze_symbol("MSFT", narrative_generator=my_generator)
```
